## Landing Layer: Trade Domain Ingestion (landing_trade_all)

- **Purpose**: Reads raw Trade domain files (`Trade.txt`, `TradeHistory.txt`, `HoldingHistory.txt`) from ADLS Gen2 Raw Zone. Proves the files are readable, appends PWG-mandated lineage metadata (`_landing_ts`, `_batch`, `_source_file`, `_run_id`), and writes them to the temporary Parquet landing zone.
- **Business Context**: PWG Pipeline - Trade Domain. Acts as the "loading dock" that isolates downstream Bronze/Silver processing from external raw storage. No data typing or business logic is applied here; everything remains as raw strings.
- **Execution Frequency**: Per Batch
- **Inputs**: 
  - `abfss://raw@.../batch{N}/Trade.txt`
  - `abfss://raw@.../batch{N}/TradeHistory.txt` (Batch 1 only)
  - `abfss://raw@.../batch{N}/HoldingHistory.txt`
- **Outputs**: 
  - `landing.trade` (Parquet, Overwrite)
  - `landing.tradehistory` (Parquet, Overwrite)
  - `landing.holdinghistory` (Parquet, Overwrite)
- **Dependencies**: **MUST RUN AFTER** Stage 0 (Raw Zone CRC/Integrity Check).

> Imported our operaitons notebook which include all the functions and all

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime

In [0]:
dbutils.widgets.text("batch_id","1")
dbutils.widgets.text("adf_link", "abfss://raw@schwabdldevsa.dfs.core.windows.net")
dbutils.widgets.text("base_path","/Volumes/charles_schwab_retailbrokerage_dev_team_lemma/landing/pwg/")

batch_id = dbutils.widgets.get("batch_id")
adf_link = dbutils.widgets.get("adf_link")
base_path = dbutils.widgets.get("base_path")


run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

In [0]:
# Schema for Trade
trade_schema_b1 = StructType([
    StructField("T_ID", StringType(), True),
    StructField("T_DTS", StringType(), True),
    StructField("T_ST_ID", StringType(), True),
    StructField("T_TT_ID", StringType(), True),
    StructField("T_IS_CASH", StringType(), True),
    StructField("T_S_SYMB", StringType(), True),
    StructField("T_QTY", StringType(), True),
    StructField("T_BID_PRICE", StringType(), True),
    StructField("T_CA_ID", StringType(), True),
    StructField("T_EXEC_NAME", StringType(), True),
    StructField("T_TRADE_PRICE", StringType(), True),
    StructField("T_CHRG", StringType(), True),
    StructField("T_COMM",StringType(), True),
    StructField("T_TAX",StringType(), True)
    ])

trade_schema_b23 = StructType([ StructField("CDC_FLAG", StringType(), True),
                                StructField("CDC_DSN", StringType(), True) 
                            ] + trade_schema_b1.fields )


# Schema for Holding history
hh_schema_b1 = StructType([
    StructField("HH_H_T_ID", StringType(), True),
    StructField("HH_T_ID", StringType(), True),
    StructField("HH_BEFORE_QTY", StringType(), True),
    StructField("HH_AFTER_QTY", StringType(), True)
])

hh_schema_b23 = StructType([
    StructField("CDC_FLAG", StringType(), True),
    StructField("CDC_DSN", StringType(), True), 
] + hh_schema_b1.fields)


# Tradehistory schema 
trade_history_schema = StructType([
    StructField("TH_T_ID", StringType(), True),
    StructField("TH_DTS", StringType(), True),
    StructField("TH_ST_ID", StringType(), True)
])


In [0]:
def load_domain(name, batch_id):
    """Loads the domain data from the given batch id and save it into parquet format"""
    # File and batch id handelling logic here
    if name == "Trade":
        if batch_id == "1":
            schema = trade_schema_b1
        else:
            schema = trade_schema_b23
        
    elif name == "HoldingHistory":
        if batch_id == "1":
            schema = hh_schema_b1
        else:
            schema = hh_schema_b23
        
    elif name == "TradeHistory" and batch_id == "1":
        schema = trade_history_schema
    
    else: 
         print(f"Skipping: Batch-{batch_id} not applicable for {name}"); 
         return None

    # Injestion logic here 
    try:
        domain_df = spark.read.format("csv")\
            .schema(schema)\
            .option("sep", "|")\
            .load(f"{adf_link}/Batch{batch_id}/{name}.txt")\
            .withColumns({
            "_landing_ts": current_timestamp(),
            "_batch": lit(batch_id),
            "_source_file": col("_metadata.file_name"),
            "_run_id" : lit(run_id)
        })
            
        # Writing to parquet format at desired location which is our base_path
        print(f"Writing {name} to parquet format")
        domain_df.write.format("parquet").mode("overwrite").save(f"{base_path}/Batch{batch_id}/{name.lower()}/")
        print(f"Writing {name} to parquet format completed")

        # Operations Logging
        # Extract the carry-forwarded _run_id from dataframe
        try:
            print("Logging.... Please Wait")
            source_count = domain_df.count()
            print(f"Source Count: {source_count}")
            target_count = spark.read.format("parquet").load(f"{base_path}/Batch{batch_id}/{name.lower()}/").count()
            print(f"Target Count: {target_count}")

            log_pipeline_recon(
                spark=spark,
                run_id=run_id,
                batch_id=batch_id,
                domain=name.upper(),
                table_name=name.lower(),
                source_layer="landing",
                target_layer="parquet",
                source_count=int(source_count),
                target_count=int(target_count)
            )

            log_audit_event(
                spark=spark,
                run_id=run_id,
                batch=batch_id,
                layer="landing",
                table_name=name.lower(),
                operation="OVERWRITE",
                rows_affected=int(target_count)
            )

            print("Done")
        except Exception as e:
            print(f"Error during operations logging: {e}")

    except Exception as e:
        print(f"Injestion Failed coz of Error: {e}")

    return None

In [0]:

# Trade file Injestion 
load_domain("Trade", batch_id)

# HoldingHistory Injestion
load_domain("HoldingHistory", batch_id)

# TradeHistory Injestion
load_domain("TradeHistory", batch_id)